# Model Output Notebook

<img style="float:center;" src="https://arcticexpansion.vse.gmu.edu/sites/arcticexpansion.vsnet.gmu.edu/files/images/header5d2.png" width=600px>

### ADCIRC-SWAN Output


### Initialize Libraries

In [1]:
import warnings;warnings.filterwarnings("ignore")
import netCDF4 as nc4;        import pandas as pd
import pathlib as pl;         import os
import numpy as np;           import xarray as xr
import geopandas as gpd;
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from functools import partial
from scipy.signal import find_peaks
from scipy.sparse import coo_matrix
from shapely.geometry import Point

In [2]:

def compute_keep_idx(sample_path, min_depth, max_depth):
    """
    Return indices of nodes whose depth is between min_depth and max_depth,
    excluding fill‐values.
    """
    ds0   = xr.open_dataset(sample_path, engine="netcdf4", mask_and_scale=False)
    depth = ds0.depth.values
    fv    = ds0.depth.encoding.get("_FillValue", None)
    mask  = np.ones_like(depth, dtype=bool)
    if fv is not None:
        mask &= (depth != fv)                  # drop missing
    mask &= (depth >= min_depth)             # deeper than shore
    mask &= (depth <= max_depth)             # shallower than cutoff
    keep_idx = np.where(mask)[0]
    ds0.close()
    return keep_idx


def open_zeta_dataset(root_path, years, keep_idx,
                      chunks={"time":1000, "node":20000}):
    """
    Read each year’s CF‑ified file (no chunks at open), preprocess it,
    then chunk and concatenate along time.
    """
    ds_list = []
    for year in years:
        path = pl.Path(root_path) / str(year) / "fort.63.cf.nc"
        # 1) Open without chunks
        ds = xr.open_dataset(path,
                             engine="netcdf4",
                             mask_and_scale=False)

        # 2) Preprocess (time mask, spin‑up drop, node slice)
        ds = preprocess(ds, year, keep_idx)

        # 3) Now apply chunking on the cleaned dataset
        ds = ds.chunk(chunks)

        ds_list.append(ds)

    # 4) Concatenate all years along time
    return xr.concat(ds_list, dim="time")

def build_adjacency(global_conn, keep_idx):
    """
    Filter the global connectivity to only triangles fully inside keep_idx,
    remap to local indices, and build a CSR adjacency matrix.
    """
    # 1) select triangles whose 3 nodes are all in keep_idx
    mask_tri = np.all(np.isin(global_conn, keep_idx), axis=1)
    tri_filt = global_conn[mask_tri]

    # 2) build a map from global -> local
    local_map = {g: i for i, g in enumerate(keep_idx)}
    # 3) remap
    remap = np.vectorize(local_map.get)(tri_filt)

    # 4) assemble edges
    row = np.hstack([remap[:,0], remap[:,1],
                     remap[:,1], remap[:,2],
                     remap[:,2], remap[:,0]])
    col = np.hstack([remap[:,1], remap[:,0],
                     remap[:,2], remap[:,1],
                     remap[:,0], remap[:,2]])
    data = np.ones_like(row, dtype=int)

    N = len(keep_idx)
    return sparse.coo_matrix((data, (row, col)), shape=(N, N)).tocsr()


def compute_keep_idx2(
    sample_path,
    min_depth,
    max_depth,
    bbox=None,
):
    """
    Return indices of nodes whose depth is between min_depth and max_depth,
    optionally also lying within a bounding box in x/y.

    Parameters
    ----------
    sample_path : str or Path
        Path to a CF‑ified fort.63 file.
    min_depth, max_depth : float
        Depth range (inclusive) in meters.
    bbox : tuple of floats (min_x, max_x, min_y, max_y), optional
        If given, further restrict to nodes with:
           min_x <= x <= max_x  AND  min_y <= y <= max_y.

    Returns
    -------
    keep_idx : np.ndarray of int
        Indices of the `node` dimension satisfying your criteria.
    """
    ds0 = xr.open_dataset(sample_path, engine="netcdf4", mask_and_scale=False)
    
    # 1) depth mask
    depth = ds0.depth.values
    fv    = ds0.depth.encoding.get("_FillValue", None)
    mask = np.ones_like(depth, dtype=bool)
    if fv is not None:
        mask &= (depth != fv)
    mask &= (depth >= min_depth)
    mask &= (depth <= max_depth)

    # 2) optional bbox mask
    if bbox is not None:
        min_x, max_x, min_y, max_y = bbox
        x = ds0.x.values
        y = ds0.y.values
        mask &= (x >= min_x) & (x <= max_x)
        mask &= (y >= min_y) & (y <= max_y)

    ds0.close()
    return np.where(mask)[0]

In [3]:
lat1, lat2 = 56, 75
lon1, lon2 = -168.5, -140

In [4]:
root = '/scratch/tmiesse/project/data4spatial'
YEARS = range(1980, 2025)
MIN_DEPTH = 0.25
MAX_DEPTH = 5
bbox = (lon1, lon2, lat1, lat2)
sample_path = f"{root}/2023/fort.93.cf.nc"
keep_idx = compute_keep_idx2(
    sample_path=sample_path,
    min_depth=MIN_DEPTH,
    max_depth=MAX_DEPTH,
    bbox=bbox
)
print(f"Keeping {len(keep_idx)}")

Keeping 189873


In [5]:
from dask.distributed import Client, LocalCluster
import math
n_workers   = int(os.environ.get("SLURM_NTASKS", "45"))
mem_per_cpu = os.environ.get("SLURM_MEM_PER_CPU", "72GB")

cluster = LocalCluster(
    n_workers=n_workers,
    threads_per_worker=1,
    processes=True,
    memory_limit=mem_per_cpu
)
client = Client(cluster)
print(client)


<Client: 'tcp://127.0.0.1:39775' processes=45 threads=45, memory=2.95 TiB>


In [6]:
def preprocess(ds):
    # 1) decode raw time → pandas
    ds = ds.assign_coords(time=pd.to_datetime(ds.time.values))
    # 2) drop anything outside that calendar year
    year = ds.time.dt.year.values[600]
    mask = ds.time.dt.year == year
    #ds = ds.sel(time=slice(f"{year}-01-01", f"{year}-12-31"))
    # 3) drop the spin‑up
    ds = ds.isel(time=mask)
    # 4) subset to shallow nodes
    ds = ds.isel(node=keep_idx)
    return ds

In [7]:
paths = [str(pl.Path(root)/str(y)/"fort.93.cf.nc") for y in YEARS]
ds_all =  xr.open_mfdataset(
            paths,
            engine="h5netcdf",            # <- use h5netcdf instead of netcdf4
            mask_and_scale=True,   # ← automatically turn fill‐values into NaN
            decode_cf=True,        # ← apply CF conventions (including _FillValue)
            preprocess=preprocess,
            combine="nested",
            concat_dim="time",
            chunks={'time':24})

times  = ds_all.time.values      # length T
ice = ds_all.iceaf.values
#z = ds_all.zeta.values.astype(float)
x_all  = ds_all.x.values                          # (n_nodes,)
y_all  = ds_all.y.values                          # (n_nodes,)


/home/tmiesse/miniforge3/envs/general/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 14.13 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [ ]:
paths = [str(pl.Path(root)/str(y)/"fort.63.cf.nc") for y in YEARS]
ds_all =  xr.open_mfdataset(
            paths,
            engine="h5netcdf",            # <- use h5netcdf instead of netcdf4
            mask_and_scale=True,   # ← automatically turn fill‐values into NaN
            decode_cf=True,        # ← apply CF conventions (including _FillValue)
            preprocess=preprocess,
            combine="nested",
            concat_dim="time",
            chunks={'time':24},
            coords="minimal")

times  = ds_all.time.values      # length T
z = ds_all.zeta.values
x_all  = ds_all.x.values                          # (n_nodes,)
y_all  = ds_all.y.values                          # (n_nodes,)


/home/tmiesse/miniforge3/envs/general/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 13.59 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [ ]:
import ruptures as rpt
da_monthly = ds_all['iceaf'].resample(time='M').max(dim='time')
z_mon      = da_monthly.values               # shape (n_months, nnode)
times      = da_monthly['time'].values       # datetime64[ns] array
nnode      = z_mon.shape[1]
x_all      = ds_all['x'].values
y_all      = ds_all['y'].values

# ── 3) Change-point detection on monthly series ─────────────────────────────
def detect_cp_monthly(y, times):
    """
    y: 1D numpy array of monthly maxima for one node
    times: 1D datetime64 array of same length
    returns: datetime64 of change-point month-end, or NaT
    """
    if np.sum(~np.isnan(y)) < 5:
        return np.datetime64('NaT')
    algo = rpt.Binseg(model='rbf').fit(y.reshape(-1,1))
    bkps = algo.predict(n_bkps=1)
    cp   = bkps[0]
    if 0 < cp < len(times):
        return times[cp]
    else:
        return np.datetime64('NaT')

# vectorize over nodes
inflection_times = np.array([detect_cp_monthly(z_mon[:, i], times)
                             for i in range(nnode)])

# ── 4) Wrap into DataArray ─────────────────────────────────────────────────
da_inflection_time = xr.DataArray(
    inflection_times,
    dims=('node',),
    coords={
        'node': ds_all['node'],
        'x':     ('node', x_all),
        'y':     ('node', y_all),
    },
    name='inflection_time'
)

# ── 5) Extract year and month as separate DataArrays ────────────────────────
da_inflection_year = da_inflection_time.dt.year.rename('inflection_year')
da_inflection_month = da_inflection_time.dt.month.rename('inflection_month')

# ── 6) Combine into a Dataset and save ─────────────────────────────────────
ds_inflection = xr.Dataset({
    'inflection_time':   da_inflection_time,
    'inflection_year':   da_inflection_year,
    'inflection_month':  da_inflection_month
}, coords=da_inflection_time.coords)

ds_inflection.to_netcdf(
    '/scratch/tmiesse/project/inflection_monthly_summary_ice.nc'
)

In [ ]:
def detect_cp_monthly(y, times):
    if np.sum(~np.isnan(y)) < 5:
        return np.datetime64('NaT')
    algo = rpt.Binseg(model='rbf').fit(y.reshape(-1,1))
    bkps = algo.predict(n_bkps=1)
    cp   = bkps[0]
    return times[cp] if 0 < cp < len(times) else np.datetime64('NaT')

# ── 1) Apply to all nodes in parallel via xarray ───────────────────────────
cp_time_da = xr.apply_ufunc(
    detect_cp_monthly,
    da_monthly,
    input_core_dims=[['time']],
    kwargs={'times': da_monthly['time'].values},
    vectorize=True,
    dask='parallelized',
    output_dtypes=['datetime64[ns]']
)

# ── 2) Rename and attach spatial coords ────────────────────────────────────
cp_time_da = cp_time_da.rename('inflection_time').assign_coords({
    'x': ('node', ds_all['x'].values),
    'y': ('node', ds_all['y'].values),
})

# ── 3) Extract year & month ────────────────────────────────────────────────
da_inf_year = cp_time_da.dt.year.rename('inflection_year')
da_inf_mon  = cp_time_da.dt.month.rename('inflection_month')

# ── 4) Bundle into a Dataset and save ─────────────────────────────────────
ds_inf = xr.Dataset({
    'inflection_time':   cp_time_da,
    'inflection_year':   da_inf_year,
    'inflection_month':  da_inf_mon
})
outpath = '/scratch/tmiesse/project/inflection_monthly_ice.nc'
ds_inf.to_netcdf(outpath)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib as mpl
ds_inf = xr.open_dataset('/scratch/tmiesse/project/inflection_monthly_summary_ice.nc')

# 2) Pull out the inflection year, month, and coordinates
inf_years  = ds_inf['inflection_year'].values    # (nnode,)
inf_months = ds_inf['inflection_month'].values   # (nnode,), ints 1–12
lon, lat    = ds_inf['x'].values, ds_inf['y'].values

# 3) Mask valid inflection‐year entries and find top‐4 years by node count
mask       = ~np.isnan(inf_years)
years      = inf_years[mask].astype(int)
unique_years, counts = np.unique(years, return_counts=True)
top4_idx        = np.argsort(counts)[::-1][:4]
selected_years  = np.sort(unique_years[top4_idx])

# 4) Build a bin‐index array for those top‐4 years
year_to_bin = {yr: i for i, yr in enumerate(selected_years)}
bin_idx     = np.full_like(inf_years, -1, dtype=int)
for yr, b in year_to_bin.items():
    bin_idx[inf_years == yr] = b
mask4 = bin_idx >= 0

# 5) Define discrete colormap and norm for the 4 bins
cmap       = plt.get_cmap('tab10', len(selected_years))
boundaries = np.arange(len(selected_years)+1) - 0.5    # e.g. [-0.5,0.5,1.5,2.5,3.5]
norm       = mpl.colors.BoundaryNorm(boundaries, len(selected_years))

# 6) Plot
fig = plt.figure(figsize=(7, 5))
ax  = plt.axes(projection=ccrs.NorthPolarStereo(central_longitude=-145))
ax.set_extent([-168, -140, 58, 71], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND,  facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')
ax.coastlines(resolution='10m', linewidth=0.5)
ax.gridlines(
    xlocs=np.arange(-180, -100, 5),
    ylocs=np.arange(  50,   90, 3),
    draw_labels=False, linewidth=0.2,
    color='black', alpha=0.3, linestyle='--'
)

# scatter only the top‐4 inflection nodes, colored by year bin
sc = ax.scatter(
    lon[mask4], lat[mask4],
    c=bin_idx[mask4],
    cmap=cmap, norm=norm,
    s=4, transform=ccrs.PlateCarree(),
    zorder=2
)

# subplot label
ax.text(0.02, 1.05, 'a.', transform=ax.transAxes,
        fontsize=12, va='top', ha='left')

# 7) Colorbar on the right
cax = fig.add_axes([0.85, 0.11, 0.02, 0.77])  # [left, bottom, width, height]
cb  = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=cax,
    orientation='vertical',
    boundaries=boundaries,
    ticks=np.arange(len(selected_years))
)
cb.set_ticklabels(selected_years.astype(int))
cb.set_label('Inflection Year', fontsize=12)

plt.savefig(
    '/scratch/tmiesse/project/figures/inflection_map_ice_updated.png',
    dpi=720, bbox_inches='tight', pad_inches=0.1
)
plt.show()

In [ ]:
unique_years, counts = np.unique(years, return_counts=True)
df = pd.DataFrame({'Inflection Year': unique_years, 'Count': counts})

# 4) Display table in the notebook
print(df.to_string(index=False))

# 5) Plot a bar chart of the distribution
plt.figure(figsize=(6, 4))
plt.bar(unique_years, counts)
plt.xlabel('Inflection Year')
plt.ylabel('Number of Nodes')
plt.title('Distribution of Inflection Years')
plt.tight_layout()
plt.show()

In [ ]:
inf_years = ds_inf['inflection_year'].values   # shape (nnode,)

# 1) Identify which years actually occurred
valid = ~np.isnan(inf_years)
unique_years = np.unique(inf_years[valid].astype(int))
nnode        = inf_years.size

# 2) Pre‐allocate (nyears_sel × 12 × nnode) and compute only those monthly maxes
monthly_max_sel = np.full((len(unique_years), 12, nnode), np.nan)
for j, yr in enumerate(unique_years):
    # select only that calendar year
    da_year = ds_all['zeta'].sel(time=ds_all.time.dt.year == yr)
    # compute its 12 monthly maxima
    da_mon  = da_year.resample(time='M').max('time')
    # load into memory for this one year
    monthly_max_sel[j, :, :] = da_mon.values   # shape (12, nnode)

In [ ]:
def detect_cp_monthly(y):
    if np.sum(~np.isnan(y)) < 6:
        return np.nan
    algo = rpt.Binseg(model='rbf').fit(y.reshape(-1,1))
    cp   = algo.predict(n_bkps=1)[0]
    return (cp + 1) if 0 < cp < 12 else np.nan
year_to_idx = {yr: i for i, yr in enumerate(unique_years)}
inf_month = np.full(nnode, np.nan)
for i in range(nnode):
    yr = inf_years[i]
    if np.isnan(yr):
        continue
    j = year_to_idx[int(yr)]
    series = monthly_max_sel[j, :, i]     # 12‐month series
    inf_month[i] = detect_cp_monthly(series)
da_out = xr.DataArray(
    inf_month,
    dims=('node',),
    coords={'node': ds_inf['node'],
            'x': ('node', x_all),
            'y': ('node', y_all)},
    name='inflection_month'
)
da_out.to_dataset().to_netcdf('/scratch/tmiesse/project/inflection_monthly.nc')

In [ ]:
months = inf_month[~np.isnan(inf_month)].astype(int)

# 3) Count occurrences for each month (1–12)
counts = np.bincount(months, minlength=13)[1:]  # skip index 0
month_labels = np.arange(1, 13)

# 4) Plot the distribution as a bar chart
plt.figure(figsize=(4, 2))
plt.bar(month_labels, counts)
plt.xlabel('Inflection Month')
plt.ylabel('Number of Nodes')
plt.title('Distribution of Inflection Months Across Nodes')
plt.xticks(month_labels)
plt.tight_layout()
plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import HypothesisTestWarning
years     = da_annual['year'].values.astype(int)  # e.g. [1980,…,2024]
nnode     = da_annual.sizes['node']
alpha     = 0.05

# 2) Pre‐allocate result arrays
pre_stat    = np.full(nnode, False, dtype=bool)
post_stat   = np.full(nnode, False, dtype=bool)
pre_p_adf   = np.full(nnode, np.nan)
pre_p_kpss  = np.full(nnode, np.nan)
post_p_adf  = np.full(nnode, np.nan)
post_p_kpss = np.full(nnode, np.nan)

def test_stationarity(arr):
    """
    Returns (is_stationary, p_adf, p_kpss) for a 1D numpy array.
    Suppresses out-of-range warnings and clamps p-values to [0,1].
    """
    # defaults if something fails
    p_adf = np.nan
    p_kpss = np.nan

    # ADFuller: stationary if p_adf < alpha
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", HypothesisTestWarning)
        try:
            p_adf = adfuller(arr, autolag='AIC')[1]
        except Exception:
            # test-stat outside MacKinnon table → strong evidence against unit root
            p_adf = 0.0

    # KPSS: stationary if p_kpss > alpha
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", HypothesisTestWarning)
        try:
            p_kpss = kpss(arr, regression='c', nlags='auto')[1]
        except Exception:
            # test-stat outside KPSS table → strong evidence of stationarity
            p_kpss = 1.0

    # clamp into valid range
    p_adf = min(max(p_adf, 0.0), 1.0)
    p_kpss = min(max(p_kpss, 0.0), 1.0)

    is_stat = (p_adf < alpha) and (p_kpss > alpha)
    return is_stat, p_adf, p_kpss

In [ ]:
# 4) Loop over nodes
for i in range(nnode):
    inf = inf_years[i]
    if np.isnan(inf):
        continue
    # find the index of the break-year in the years array
    cp_idx = np.where(years == int(inf))[0]
    if cp_idx.size == 0 or cp_idx[0] == 0 or cp_idx[0] >= len(years):
        continue
    cp = cp_idx[0]
    series = da_annual[:, i].values   # length = len(years)

    # split
    pre  = series[:cp]
    post = series[cp:]

    # require at least 3 points to test
    if len(pre) >= 3:
        s, p1, p2 = test_stationarity(pre)
        pre_stat[i], pre_p_adf[i], pre_p_kpss[i] = s, p1, p2
    if len(post) >= 3:
        s, p1, p2 = test_stationarity(post)
        post_stat[i], post_p_adf[i], post_p_kpss[i] = s, p1, p2

In [ ]:
# 5) Wrap results as DataArrays and save
coords = {
    'node': da_annual['node'],
    'x': ('node', x_all),
    'y': ('node', y_all),
}

ds_out = xr.Dataset({
    'pre_stationary':    (('node',), pre_stat,   {'long_name': 'pre-inflection stationary?'}),
    'post_stationary':   (('node',), post_stat,  {'long_name': 'post-inflection stationary?'}),
    'pre_p_adf':         (('node',), pre_p_adf,  {'long_name': 'ADF p-value (pre)'}),
    'pre_p_kpss':        (('node',), pre_p_kpss, {'long_name': 'KPSS p-value (pre)'}),
    'post_p_adf':        (('node',), post_p_adf, {'long_name': 'ADF p-value (post)'}),
    'post_p_kpss':       (('node',), post_p_kpss,{'long_name': 'KPSS p-value (post)'}),
}, coords=coords)

outpath = '/scratch/tmiesse/project/inflection_stationarity.nc'
ds_out.to_netcdf(outpath)

In [ ]:
ds_stat    = xr.open_dataset('/scratch/tmiesse/project/inflection_stationarity.nc')
pre_stat   = ds_stat['pre_stationary'].values   # boolean (node,)
post_stat  = ds_stat['post_stationary'].values  # boolean (node,)
lon        = ds_stat['x'].values
lat        = ds_stat['y'].values

# 2) Build category array: 0=pre-only, 1=post-only, 2=both
mask      = pre_stat | post_stat
cat       = np.full(pre_stat.shape, -1, dtype=int)
cat[(pre_stat) & (~post_stat)] = 0
cat[(post_stat) & (~pre_stat)] = 1
cat[(pre_stat) & (post_stat)]  = 2

# 3) Define labels and colors
labels = ['Pre-stationary only', 'Post-stationary only', 'Both stationary']
colors = ['tab:blue',          'tab:orange',           'tab:green']

# 4) Plot
fig = plt.figure(figsize=(4,4))
ax  = plt.axes(projection=ccrs.NorthPolarStereo(central_longitude=-145))
ax.set_extent([-169, -140, 58, 71], crs=ccrs.PlateCarree())

# base map
ax.add_feature(cfeature.LAND,   facecolor='lightgray')
ax.add_feature(cfeature.OCEAN,  facecolor='aliceblue')
ax.coastlines(resolution='10m', linewidth=0.5)

gl = ax.gridlines(
    xlocs=np.arange(-180, -100, 5),
    ylocs=np.arange(  50,   90, 3),
    draw_labels=True, linewidth=0.3,
    color='black', alpha=0.5, linestyle='--'
)
gl.xlabels_top    = False
gl.ylabels_right  = False

# scatter each category
for c, label, color in zip([0,1,2], labels, colors):
    idx = (cat == c)
    ax.scatter(
        lon[idx], lat[idx],
        color=color,
        s=5,
        transform=ccrs.PlateCarree(),
        label=label,
        edgecolor='k', linewidth=0.01
    )

# legend
ax.legend(loc='upper left', frameon=False, title='Stationarity')

plt.title('Pre- vs Post-Inflection Stationarity', fontsize=14)
plt.show()

In [ ]:
from scipy.stats import genextreme
from scipy.stats import gumbel_r
ds_stat  = xr.open_dataset('/scratch/tmiesse/project/inflection_stationarity.nc')
pre_stat  = ds_stat['pre_stationary'].values    # boolean (node,)
post_stat = ds_stat['post_stationary'].values   # boolean (node,)

# 2) Prepare output arrays
rp_pre_10  = np.full(nnode, np.nan)
rp_pre_50  = np.full(nnode, np.nan)
rp_pre_100 = np.full(nnode, np.nan)
rp_pre_500 = np.full(nnode, np.nan)
rp_post_10  = np.full(nnode, np.nan)
rp_post_50  = np.full(nnode, np.nan)
rp_post_100 = np.full(nnode, np.nan)
rp_post_500 = np.full(nnode, np.nan)
year_to_idx = {yr: idx for idx, yr in enumerate(years)}

In [ ]:
for i in range(nnode):
    yr = inf_years[i]
    if np.isnan(yr):
        continue

    cp = year_to_idx.get(int(yr), None)
    if cp is None or cp == 0 or cp >= len(years):
        continue

    # — Pre-inflection Gumbel fit —
    if pre_stat[i]:
        data_pre  = da_annual[:cp, i].values
        valid_pre = data_pre[np.isfinite(data_pre)]
        if len(valid_pre) >= 20:
            # gumbel_r.fit returns (loc, scale)
            loc, scale   = gumbel_r.fit(valid_pre)
            rp_pre_10[i]  = gumbel_r.ppf(1 - 1/10, loc=loc, scale=scale)
            rp_pre_50[i]  = gumbel_r.ppf(1 - 1/50, loc=loc, scale=scale)
            rp_pre_100[i] = gumbel_r.ppf(1 - 1/100, loc=loc, scale=scale)
            rp_pre_500[i] = gumbel_r.ppf(1 - 1/500, loc=loc, scale=scale)

    # — Post-inflection Gumbel fit —
    if post_stat[i]:
        data_post   = da_annual[cp:, i].values
        valid_post  = data_post[np.isfinite(data_post)]
        if len(valid_post) >= 20:
            loc2, scale2 = gumbel_r.fit(valid_post)
            rp_post_10[i]  = gumbel_r.ppf(1 - 1/10, loc=loc2, scale=scale2)
            rp_post_50[i]  = gumbel_r.ppf(1 - 1/50, loc=loc2, scale=scale2)
            rp_post_100[i] = gumbel_r.ppf(1 - 1/100, loc=loc2, scale=scale2)
            rp_post_500[i] = gumbel_r.ppf(1 - 1/500, loc=loc2, scale=scale2)

In [ ]:
ds_out = xr.Dataset({
    'rp_pre_10':  (('node',), rp_pre_10,  {'units':'m','long_name':'10-yr return level (pre)'}),
    'rp_pre_50':  (('node',), rp_pre_50,  {'units':'m','long_name':'50-yr return level (pre)'}),
    'rp_pre_100': (('node',), rp_pre_100, {'units':'m','long_name':'100-yr return level (pre)'}),
    'rp_pre_500': (('node',), rp_pre_500, {'units':'m','long_name':'500-yr return level (pre)'}),
    'rp_post_10':  (('node',), rp_post_10,  {'units':'m','long_name':'10-yr return level (post)'}),
    'rp_post_50':  (('node',), rp_post_50,  {'units':'m','long_name':'50-yr return level (post)'}),
    'rp_post_100': (('node',), rp_post_100, {'units':'m','long_name':'100-yr return level (post)'}),
    'rp_post_500': (('node',), rp_post_500, {'units':'m','long_name':'500-yr return level (post)'}),
}, coords={
    'node': ('node', da_annual['node'].values),
    'x':    ('node', x_all),
    'y':    ('node', y_all),
})

outpath = '/scratch/tmiesse/project/inflection_returnlevels.nc'
ds_out.to_netcdf(outpath)

In [ ]:
fields = [
    (r'Pre-Inflection $T_{10}$', ds_out['rp_pre_10'].values),
    (r'Pre-Inflection $T_{50}$', ds_out['rp_pre_50'].values),
    (r'Pre-Inflection $T_{100}$', ds_out['rp_pre_100'].values),
    (r'Post-Inflection $T_{10}$', ds_out['rp_post_10'].values),
    (r'Post-Inflection $T_{50}$', ds_out['rp_post_50'].values),
    (r'Post-Inflection $T_{100}$', ds_out['rp_post_100'].values),
]
ds = xr.open_dataset('/scratch/tmiesse/project/inflection_returnlevels.nc')
x, y = ds['x'].values, ds['y'].values
labels = ['b.', 'c.', 'd.', 'e.', 'f.','g.']
# colormap
n_bins = 7
bins   = np.linspace(0, 3.5, n_bins+1)
cmap   = plt.get_cmap('rainbow', n_bins)
norm   = mpl.colors.BoundaryNorm(bins, n_bins)

fig, axes = plt.subplots(
    2, 3,
    figsize=(12, 7),
    subplot_kw={'projection': ccrs.NorthPolarStereo(central_longitude=-145)},
    gridspec_kw={
        'left':   0.02,
        'right':  0.78,
        'top':    0.98,
        'bottom': 0.02,
        'wspace': 0.02,
        'hspace': 0.0    # zero vertical space
    }
)

# plot panels

for ax, (title, data), lab in zip(axes.flat, fields, labels):
    ax.set_extent([-169, -140, 58, 71], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,   facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN,  facecolor='aliceblue')
    ax.coastlines(resolution='10m', linewidth=0.15)
    ax.gridlines(
        xlocs=np.arange(-180, -100, 5),
        ylocs=np.arange(50, 90, 3),
        draw_labels=False, linewidth=0.2,
        color='black', alpha=0.3, linestyle='--'
    )
    mask = ~np.isnan(data)
    ax.scatter(
        x[mask], y[mask],
        c=data[mask],
        cmap=cmap, norm=norm,
        s=5,
        transform=ccrs.PlateCarree()
    )
    ax.set_title(title, fontsize=11, pad=4)

    # add subplot label
    ax.text(
        0.01, 1.05, lab,
        transform=ax.transAxes,
        fontsize=10, fontweight=None,
        va='top', ha='left'
    )

# single vertical colorbar
cax = fig.add_axes([0.80, 0.06, 0.01, 0.885])
cb = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=cax,
    orientation='vertical',
    boundaries=bins,
    ticks=bins
)
cb.set_label('WSE [m at MSL]')
plt.savefig(
    '/scratch/tmiesse/project/figures/return_period_mapsv2.png',
    dpi=720, bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
ds = xr.open_dataset('/scratch/tmiesse/project/inflection_returnlevels.nc')
x, y = ds['x'].values, ds['y'].values

# — define the 9 panels —
fields_pre = [
    (r'Pre-Inflection $T_{10}$', ds['rp_pre_10'].values),
    (r'Pre-Inflection $T_{50}$', ds['rp_pre_50'].values),
    (r'Pre-Inflection $T_{100}$', ds['rp_pre_100'].values),
]
fields_post = [
    (r'Post-Inflection $T_{10}$', ds['rp_post_10'].values),
    (r'Post-Inflection $T_{50}$', ds['rp_post_50'].values),
    (r'Post-Inflection $T_{100}$', ds['rp_post_100'].values),
]
fields_diff = [
    (r'$\Delta T_{10}$', ds['rp_post_10'].values - ds['rp_pre_10'].values),
    (r'$\Delta T_{50}$', ds['rp_post_50'].values - ds['rp_pre_50'].values),
    (r'$\Delta T_{100}$', ds['rp_post_100'].values - ds['rp_pre_100'].values),
]
all_fields = fields_pre + fields_post + fields_diff
labels    = ['a.', 'b.', 'c.', 'd.', 'e.', 'f.', 'g.', 'h.', 'i.']

# — colormaps & norms —
bins_pp   = np.linspace(0, 3.5, 8)
cmap_pp   = plt.get_cmap('rainbow', len(bins_pp)-1)
norm_pp   = mpl.colors.BoundaryNorm(bins_pp, len(bins_pp)-1)

max_diff  = np.nanmax(np.abs(fields_diff[0][1])) or 1.0
cmap_diff = plt.get_cmap('bwr', 8)
vmin,vmax = -1.5,1.5
norm_diff = mpl.colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

# — make figure with negative hspace —
fig, axes = plt.subplots(
    3, 3, figsize=(12, 12),
    subplot_kw={'projection': ccrs.NorthPolarStereo(central_longitude=-145)},
    gridspec_kw={
        'left':   0.02,
        'right':  0.78,
        'top':    0.99,
        'bottom': 0.01,
        'wspace': 0.01,
        'hspace': -0.4    # negative! shrinks vertical gaps
    }
)

for idx, ax in enumerate(axes.flat):
    title, data = all_fields[idx]
    lab         = labels[idx]

    ax.set_extent([-169, -140, 58, 71], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,   facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN,  facecolor='aliceblue')
    ax.coastlines(resolution='10m', linewidth=0.1)
    ax.gridlines(
        xlocs=np.arange(-180, -100, 5),
        ylocs=np.arange(50, 90, 3),
        draw_labels=False, linewidth=0.2,
        color='black', alpha=0.3, linestyle='--'
    )

    mask = ~np.isnan(data)
    if idx < 6:
        sc = ax.scatter(
            x[mask], y[mask], c=data[mask],
            cmap=cmap_pp, norm=norm_pp,
            s=5, transform=ccrs.PlateCarree()
        )
    else:
        sc = ax.scatter(
            x[mask], y[mask], c=data[mask],
            cmap=cmap_diff, norm=norm_diff,
            s=5, transform=ccrs.PlateCarree()
        )

    ax.set_title(title, fontsize=11, pad=2)
    ax.text(
        0.02, 1.05, lab,
        transform=ax.transAxes,
        fontsize=10, va='top'
    )

# — colorbar for pre/post (rows 1–2) —
cax1 = fig.add_axes([0.80, 0.385, 0.015, 0.5])
cb1  = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm_pp, cmap=cmap_pp),
    cax=cax1, orientation='vertical',
    boundaries=bins_pp, ticks=bins_pp
)
cb1.set_label('WSE [m at MSL]', fontsize=12)

# — colorbar for difference (row 3) —
cax2 = fig.add_axes([0.80, 0.115, 0.015, 0.235])
cb2  = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm_diff, cmap=cmap_diff),
    cax=cax2, orientation='vertical',
    ticks=np.linspace(vmin, vmax, 5)
)
cb2.set_label('Δ WSE [m]', fontsize=12)
plt.savefig(
    '/scratch/tmiesse/project/figures/return_period+diff_mapsv2.png',
    dpi=720, bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
ds = xr.open_dataset('/scratch/tmiesse/project/inflection_returnlevels.nc')
x, y = ds['x'].values, ds['y'].values

# Define pre- and post-inflection fields
fields_pre = [
    (r'Pre-Inflection $T_{10}$',  ds['rp_pre_10'].values),
    (r'Pre-Inflection $T_{50}$',  ds['rp_pre_50'].values),
    (r'Pre-Inflection $T_{100}$', ds['rp_pre_100'].values),
]
fields_post = [
    (r'Post-Inflection $T_{10}$',  ds['rp_post_10'].values),
    (r'Post-Inflection $T_{50}$',  ds['rp_post_50'].values),
    (r'Post-Inflection $T_{100}$', ds['rp_post_100'].values),
]

# Compute percent difference: (post - pre) / pre * 100
fields_diff = []
for (label_pre, arr_pre), (label_post, arr_post) in zip(fields_pre, fields_post):
    # percent change
    pct = (arr_post - arr_pre) / arr_pre * 100
    # extract return period from label_pre
    Tn = label_pre.split('$T_{')[-1].strip('}$')
    fields_diff.append((rf'$\Delta T_{{{Tn}}}$', pct))

# Combine all fields and labels
all_fields = fields_pre + fields_post + fields_diff
labels = ['a.', 'b.', 'c.', 'd.', 'e.', 'f.', 'g.', 'h.', 'i.']

# Colormaps & norms for pre/post
bins_pp = np.linspace(0, 3.5, 8)
cmap_pp = plt.get_cmap('rainbow', len(bins_pp)-1)
norm_pp = mpl.colors.BoundaryNorm(bins_pp, len(bins_pp)-1)

# Colormap & norm for percent difference
# determine symmetric range
all_pct = np.concatenate([pct for _, pct in fields_diff])
max_pct = np.nanpercentile(np.abs(all_pct), 99)  # 99th percentile to avoid outliers
vmin, vmax = -50,50
cmap_diff = plt.get_cmap('coolwarm', 12)
norm_diff = mpl.colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
ticks_diff = np.linspace(vmin, vmax,11)

# Create 3×3 panel with minimal whitespace
fig, axes = plt.subplots(
    3, 3, figsize=(12, 12),
    subplot_kw={'projection': ccrs.NorthPolarStereo(central_longitude=-145)},
    gridspec_kw={
        'left': 0.02, 'right': 0.78,
        'top': 0.99, 'bottom': 0.01,
        'wspace': 0.01, 'hspace': -0.4
    }
)

# Plot each panel
for idx, ax in enumerate(axes.flat):
    title, data = all_fields[idx]
    lab = labels[idx]

    ax.set_extent([-169, -140, 58, 71], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')
    ax.coastlines(resolution='10m', linewidth=0.1)
    ax.gridlines(
        xlocs=np.arange(-180, -100, 5),
        ylocs=np.arange(50, 90, 3),
        draw_labels=False, linewidth=0.2,
        color='black', alpha=0.3, linestyle='--'
    )

    mask = ~np.isnan(data)
    if idx < 6:
        sc = ax.scatter(
            x[mask], y[mask], c=data[mask],
            cmap=cmap_pp, norm=norm_pp,
            s=5, transform=ccrs.PlateCarree()
        )
    else:
        sc = ax.scatter(
            x[mask], y[mask], c=data[mask],
            cmap=cmap_diff, norm=norm_diff,
            s=5, transform=ccrs.PlateCarree()
        )

    ax.set_title(title, fontsize=11, pad=2)
    ax.text(0.02, 1.05, lab, transform=ax.transAxes, fontsize=10, va='top')

# Colorbar for pre/post (rows 1–2)
cax1 = fig.add_axes([0.80, 0.385, 0.015, 0.5])
cb1 = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm_pp, cmap=cmap_pp),
    cax=cax1, orientation='vertical',
    boundaries=bins_pp, ticks=bins_pp
)
cb1.set_label('WSE [m at MSL]', fontsize=12)

# Colorbar for percent difference (row 3)
cax2 = fig.add_axes([0.80, 0.115, 0.015, 0.235])
cb2 = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm_diff, cmap=cmap_diff),
    cax=cax2, orientation='vertical',
    ticks=ticks_diff
)
cb2.set_ticklabels([f"{t:.0f}%" for t in ticks_diff])
cb2.set_label(r'$ \Delta T$ [%]', fontsize=12)

plt.savefig(
    '/scratch/tmiesse/project/figures/return_period_plus_pctdiff_maps.png',
    dpi=720, bbox_inches='tight', pad_inches=0.1
)
plt.show()

In [ ]:
ds = xr.open_dataset('/scratch/tmiesse/project/inflection_returnlevels.nc')
x, y = ds['x'].values, ds['y'].values

# Define pre- and post-inflection fields
fields_pre = [
    (r'Pre-Inflection $T_{10}$',  ds['rp_pre_10'].values),
    (r'Pre-Inflection $T_{50}$',  ds['rp_pre_50'].values),
    (r'Pre-Inflection $T_{100}$', ds['rp_pre_100'].values),
]
fields_post = [
    (r'Post-Inflection $T_{10}$',  ds['rp_post_10'].values),
    (r'Post-Inflection $T_{50}$',  ds['rp_post_50'].values),
    (r'Post-Inflection $T_{100}$', ds['rp_post_100'].values),
]

# Compute percent difference
fields_diff = []
for (label_pre, arr_pre), (label_post, arr_post) in zip(fields_pre, fields_post):
    pct = (arr_post - arr_pre) / arr_pre * 100
    Tn = label_pre.split('$T_{')[-1].strip('}$')
    fields_diff.append((rf'$\Delta T_{{{Tn}}}$ (%)', pct))

# Combine all fields
all_fields = fields_pre + fields_post + fields_diff
labels    = ['a.', 'b.', 'c.', 'd.', 'e.', 'f.', 'g.', 'h.', 'i.']

# Colormaps & norms for pre/post
bins_pp = np.linspace(0, 3.5, 8)
cmap_pp = plt.get_cmap('rainbow', len(bins_pp)-1)
norm_pp = mpl.colors.BoundaryNorm(bins_pp, len(bins_pp)-1)

# --- UPDATED: Colormap & norm for percent difference (10% bins) ---
ticks_diff = np.arange(-20, 51, 10)                # [-50, -40, ..., 40, 50]
cmap_diff  = plt.get_cmap('Spectral', len(ticks_diff)-1)
norm_diff  = mpl.colors.BoundaryNorm(ticks_diff, len(ticks_diff)-1)

# Create panel
fig, axes = plt.subplots(
    3, 3, figsize=(12, 12),
    subplot_kw={'projection': ccrs.NorthPolarStereo(central_longitude=-145)},
    gridspec_kw={'left':0.02,'right':0.78,'top':0.99,'bottom':0.01,'wspace':0.01,'hspace':-0.4}
)

# Plot each subplot
for idx, ax in enumerate(axes.flat):
    title, data = all_fields[idx]
    lab          = labels[idx]

    ax.set_extent([-169, -140, 58, 71], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')
    ax.coastlines(resolution='10m', linewidth=0.1)
    ax.gridlines(xlocs=np.arange(-180,-100,5), ylocs=np.arange(50,90,3),
                 draw_labels=False, linewidth=0.2, color='black', alpha=0.3, linestyle='--')

    mask = ~np.isnan(data)
    if idx < 6:
        sc = ax.scatter(x[mask], y[mask], c=data[mask],
                        cmap=cmap_pp, norm=norm_pp,
                        s=5, transform=ccrs.PlateCarree())
    else:
        sc = ax.scatter(x[mask], y[mask], c=data[mask],
                        cmap=cmap_diff, norm=norm_diff,
                        s=5, transform=ccrs.PlateCarree())

    ax.set_title(title, fontsize=11, pad=2)
    ax.text(0.02, 1.05, lab, transform=ax.transAxes, fontsize=10, va='top')

# Colorbar for pre/post (rows 1–2)
cax1 = fig.add_axes([0.80, 0.385, 0.015, 0.5])
cb1  = fig.colorbar(mpl.cm.ScalarMappable(norm=norm_pp, cmap=cmap_pp),
                    cax=cax1, orientation='vertical',
                    boundaries=bins_pp, ticks=bins_pp)
cb1.set_label('WSE [m at MSL]', fontsize=12)

# Colorbar for percent difference (row 3)
cax2 = fig.add_axes([0.80, 0.115, 0.015, 0.235])
cb2  = fig.colorbar(mpl.cm.ScalarMappable(norm=norm_diff, cmap=cmap_diff),
                    cax=cax2, orientation='vertical',
                    boundaries=ticks_diff, ticks=ticks_diff)
cb2.set_label('Percent ΔT (%)', fontsize=12)

plt.savefig('/scratch/tmiesse/project/figures/return_period_plus_pctdiff_maps.png',
            dpi=720, bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from scipy.stats import spearmanr
import pymannkendall as mk

# ── 1) Load your annual‐max DataArray (year × node) ─────────────────────────
# If you have it on disk, uncomment these two lines:
# ds_ann    = xr.open_dataset('/scratch/tmiesse/project/annual_max_zeta.nc')
# da_annual = ds_ann['annual_max_zeta']
# Otherwise, assume da_annual is already in memory

years    = da_annual['year'].values.astype(int)
x_nodes  = da_annual['x'].values
y_nodes  = da_annual['y'].values
nnode    = da_annual.sizes['node']

# ── 2) Load inflection years ────────────────────────────────────────────────
ds_inf     = xr.open_dataset('/scratch/tmiesse/project/inflection_annual.nc')
inf_years  = ds_inf['inflection_year'].values  # shape (node,)

# ── 3) Identify non‐stationary & significant nodes (pre/post) ─────────────
alpha = 0.05

def is_stationary(arr):
    """ADF (p<α) & KPSS (p>α) → stationary."""
    arr = arr[np.isfinite(arr)]
    if len(arr) < 5: 
        return False
    try:
        p_adf  = adfuller(arr, autolag='AIC')[1]
    except:
        p_adf = 0.0
    try:
        p_kpss = kpss(arr, regression='c', nlags='auto')[1]
    except:
        p_kpss = 1.0
    return (p_adf < alpha) and (p_kpss > alpha)

def trend_significant(arr):
    """Spearman + Mann‐Kendall p<α → significant trend (non‐stationary)."""
    arr = arr[np.isfinite(arr)]
    if len(arr) < 5:
        return False
    _, p_sp = spearmanr(np.arange(len(arr)), arr)
    try:
        p_mk = mk.original_test(arr).p
    except:
        p_mk = 1.0
    return (p_sp < alpha) and (p_mk < alpha)

pre_sig  = np.zeros(nnode, bool)
post_sig = np.zeros(nnode, bool)

for i in range(nnode):
    yr = inf_years[i]
    if np.isnan(yr):
        continue
    # find index of inflection year in your years array
    idx = np.where(years == int(yr))[0]
    if idx.size == 0 or idx[0] == 0:
        continue
    cp = idx[0]
    series = da_annual[:, i].values

    pre  = series[:cp]
    post = series[cp:]
    # check stationarity and trend
    if (not is_stationary(pre)) and trend_significant(pre):
        pre_sig[i] = True
    if (not is_stationary(post)) and trend_significant(post):
        post_sig[i] = True

# ── 4) Fit linear trends (mm yr⁻¹) on those flagged segments ────────────────
slope_pre  = np.full(nnode, np.nan)
slope_post = np.full(nnode, np.nan)

for i in range(nnode):
    yr = inf_years[i]
    if np.isnan(yr):
        continue
    idx = np.where(years == int(yr))[0]
    if idx.size == 0 or idx[0] == 0:
        continue
    cp = idx[0]
    series = da_annual[:, i].values

    if pre_sig[i]:
        xp = years[:cp]
        yp = series[:cp]
        mask = np.isfinite(yp)
        if mask.sum() >= 2:
            m, _ = np.polyfit(xp[mask], yp[mask], 1)
            slope_pre[i] = m * 1000  # convert m/yr → mm/yr

    if post_sig[i]:
        xp = years[cp:]
        yp = series[cp:]
        mask = np.isfinite(yp)
        if mask.sum() >= 2:
            m, _ = np.polyfit(xp[mask], yp[mask], 1)
            slope_post[i] = m * 1000


In [ ]:

# ── 5) Define bins & colormap ────────────────────────────────────────────────
all_slopes = np.concatenate([
    slope_pre[np.isfinite(slope_pre)],
    slope_post[np.isfinite(slope_post)]
])
bins = np.linspace(vmin, vmax, 9)  # 5 bins
cmap = plt.get_cmap('PiYG', len(bins)-1)
norm = mpl.colors.BoundaryNorm(bins, len(bins)-1)

# 4) Plot side‐by‐side trend maps
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    sharey=True,
    subplot_kw={'projection': ccrs.NorthPolarStereo(central_longitude=-145)}
)

# Tighter layout: minimal space between plots
fig.subplots_adjust(left=0.04, right=0.88, top=0.96, bottom=0.04, wspace=-0.25)

titles = ['Pre-Inflection Trend (mm yr⁻¹)', 'Post-Inflection Trend (mm yr⁻¹)']
slopes = [slope_pre, slope_post]

for ax, trend, title in zip(axes, slopes, titles):
    # background
    ax.scatter(x_nodes, y_nodes, c='lightgray', s=2, transform=ccrs.PlateCarree())
    # overlay
    mask = np.isfinite(trend)
    sc = ax.scatter(
        x_nodes[mask], y_nodes[mask],
        c=trend[mask], cmap=cmap, norm=norm,
        s=20, edgecolor='k', linewidth=0.01,
        transform=ccrs.PlateCarree()
    )
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')
    ax.coastlines(resolution='10m', linewidth=0.15)
    ax.gridlines(
        xlocs=np.arange(-180, -100, 5),
        ylocs=np.arange(50, 90, 3),
        color='black', linestyle='--', linewidth=0.3, alpha=0.5,
        transform=ccrs.PlateCarree()
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Longitude')

axes[0].set_ylabel('Latitude')

# 5) Vertical colorbar on right
cax = fig.add_axes([0.82, 0.04, 0.01, 0.915])  # adjust for new right margin
cb = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=cax,
    orientation='vertical',
    boundaries=bins,
    ticks=bins
)
cb.set_label('Trend [mm yr⁻¹]', fontsize=12)

plt.show()

In [ ]:
da_annual.data.shape